In [ ]:
import json
import re
from pathlib import Path
import csv
from collections import defaultdict

# ======== 配置：改成你的路径 ========
INPUT_DIR = Path("D:\\VOCs钢铁\\数据产出\\kgdemoup\\demo1\\metadata_pdf")          # 存放 *.txt 的文件夹
PROCESSED_INDEX_JSON = Path("D:\\VOCs钢铁\\数据产出\\kgdemoup\\demo1\\progress.json")  # 形如 ["1_1999_71_chunk_9", ...]
OUTPUT_CSV = Path("D:\\VOCs钢铁\\数据产出\\kgdemoup\\demo1\\processing_status.csv")
# ===================================

CHUNK_PAT = re.compile(r"^(?P<stem>.+?)_chunk_(?P<idx>\d+)$")

def load_processed_ids(p: Path):
    """兼容：纯 JSON 列表 或 JSONL（每行一个 JSON 对象里含 doc_id）"""
    if p.suffix.lower() == ".jsonl":
        stems = defaultdict(list)
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue
                # 常见字段名尝试
                doc_id = obj.get("doc_id") or obj.get("DOC_ID") or obj.get("id")
                if not doc_id or not isinstance(doc_id, str):
                    continue
                m = CHUNK_PAT.match(doc_id)
                if m:
                    stems[m.group("stem")].append(int(m.group("idx")))
        return stems

    # 普通 JSON 列表：["xxx_chunk_0", "xxx_chunk_1", ...]
    with p.open("r", encoding="utf-8") as f:
        data = json.load(f)
    stems = defaultdict(list)
    for s in data:
        if not isinstance(s, str):
            continue
        m = CHUNK_PAT.match(s)
        if m:
            stems[m.group("stem")].append(int(m.group("idx")))
    return stems

def main():
    # 1) 所有 txt 文件名（不含后缀）
    all_txt_stems = {fp.stem for fp in INPUT_DIR.glob("*.pdf")}

    # 2) 已处理的分块 -> 聚合为 {stem: [idx...]}
    processed = load_processed_ids(PROCESSED_INDEX_JSON)
    processed_stems = set(processed.keys())

    # 3) 统计
    already = sorted(all_txt_stems & processed_stems)
    notyet  = sorted(all_txt_stems - processed_stems)

    print(f"TXT 总数: {len(all_txt_stems)}")
    print(f"已处理的 TXT 数: {len(already)}")
    print(f"未处理的 TXT 数: {len(notyet)}\n")

    if already:
        print("已处理示例（前10个）：")
        for s in already[:10]:
            print(f"  {s}  (chunks={len(set(processed[s]))}, ids={sorted(set(processed[s]))[:5]}{' ...' if len(set(processed[s]))>5 else ''})")
        print()

    if notyet:
        print("未处理文件（前20个）：")
        for s in notyet[:20]:
            print(f"  {s}")
        if len(notyet) > 20:
            print("  ...")
        print()

    # 4) 导出 CSV 明细
    with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["txt_stem", "status", "num_chunks", "chunk_ids_sorted"])
        for stem in sorted(all_txt_stems):
            ids = sorted(set(processed.get(stem, [])))
            status = "processed" if stem in processed else "not_processed"
            writer.writerow([stem, status, len(ids), " ".join(map(str, ids))])

    print(f"明细已保存：{OUTPUT_CSV.resolve()}")

if __name__ == "__main__":
    main()

TXT 总数: 191
已处理的 TXT 数: 1
未处理的 TXT 数: 190

已处理示例（前10个）：
  1_1999_71  (chunks=9, ids=[0, 1, 2, 3, 4] ...)

未处理文件（前20个）：
  1_2001_67
  1_2007_68
  1_2009_70
  1_2011_65
  1_2011_72
  1_2013_64
  1_2014_75
  1_2015_47
  1_2017_57
  1_2018_69
  1_2019_16
  1_2019_35
  1_2019_41
  1_2019_62
  1_2019_77
  1_2020_1
  1_2020_12
  1_2020_24
  1_2020_33
  1_2020_4
  ...

明细已保存：D:\VOCs钢铁\demo1\processing_status.csv
